# Export for Android

Export the trained detector to LiteRT/TFLite. The export is saved in the Drive project folder so it can be copied into the Android app later. Confirm the output tensor layout before implementing the final Android adapter.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TrayMachi/indonesia-license-plate-model.git"
REPO_DIR = Path("/content/indonesia-license-plate-model")
if not (REPO_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%pip install -q -r /content/indonesia-license-plate-model/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/indonesia-license-plate-model")
CHECKPOINT = DRIVE_ROOT / "runs" / "plate-detector" / "weights" / "best.pt"
EXPORT_DIR = DRIVE_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Update CHECKPOINT; file not found: {CHECKPOINT}")

In [ ]:
from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
exported_path = model.export(format="tflite", imgsz=640, int8=False, nms=False)
exported_path = Path(exported_path)
print("Exported model:", exported_path)
print("Input size: 640x640 RGB, float32 [0, 1] by default")

In [ ]:
import shutil

final_path = EXPORT_DIR / exported_path.name
if exported_path.resolve() != final_path.resolve():
    shutil.copy2(exported_path, final_path)
print("Android artifact:", final_path)
print("Inspect the export metadata and tensor shapes before integrating it with LiteRT.")

The Android client should use the same letterbox parameters as src/preprocess.py, then apply model-specific output decoding followed by the NMS helpers in src/postprocess.py. Quantized exports can be added later after a float32 baseline is verified.